# Perception Test

In [1]:
# ASA Imports
# Notebook Specifics / Temporary Functions
#
import logging
from pathlib import Path

from asa._tools.custom_logging import setup_logging
from asa.affect_model.belief import AffectModel
from asa.affect_model.folding import Assign, ConfidenceWeighted
from asa.core.affect import AffectEvidence, AffectState, AffectVector, Target, Utterance, utc_now
from asa.core.observers import Event, Observers
from asa.core.representations import EKMAN6, AffectRepresentation
from asa.perception.decode_keyword import EKMAN6_KEYWORDS, KeywordDecoder
from asa.perception.text_console import TextConsole
from asa.runtime import run_agent

setup_logging(level="DEBUG")
log = logging.getLogger("asa.observer.debug")

def read_or_end(prompt: str) -> str:
    """Notebook stand-in for Ctrl-D — no frontend here can send a real EOF."""
    text = input(prompt)
    if text.strip() == ":q":
        raise EOFError
    return text

def axes(vector: AffectVector | None) -> dict[str, float] | None:
    """Non-rest axes only, rounded — a full six-axis dict at full precision is unreadable.

    **`None` in, `None` out, and not an empty dict.** A source that does not declare its intent is
    making no claim; an empty dict is the claim that every axis is at rest. `Utterance.intended` is
    `None` for a person typing at a console and populated for a generated or benchmark source, so
    collapsing the two would report ground truth where there is none — the unmeasured-versus-absent
    mistake that keeps BRIGHTER's disgust as NaN rather than zero-filling it.

    Three decimals rather than two so that decay stays visible between folds: the difference
    between a belief of 0.625 and one that has aged to 0.624 is the whole point of watching a
    replay, and two decimals hides it.

    Filters against zero rather than the representation's `rest`. They coincide for both declared
    representations, and this is a debug log rather than a record — but a representation resting
    off zero would show every axis, so do not carry this into the recorder.
    """
    if vector is None:
        return None
    return {str(k): round(magnitude, 3) for k, magnitude in vector.values.items() if magnitude}


def log_event(event: Event) -> None:
    """Temporary stand-in for the recorder — schema, time, id, then the payload.

    Every branch keeps the same first three columns so the lines read as a table. A state has no
    id column of its own: it is the belief at an instant rather than a record pointing at another.
    """
    when = event.at.strftime("%H:%M:%S.%f")[:-3]

    if isinstance(event, Utterance):
        log.debug("%-12s %s  %s  %-5s %-18s %r intended=%s",
                  event.schema, when, event.id, "", event.source, event.text,
                  axes(event.intended))
    elif isinstance(event, AffectEvidence):
        log.debug("%-12s %s  %s  %-5s %-18s %s",
                  event.schema, when, event.of_input, event.target, event.source,
                  axes(event.affect))
    elif isinstance(event, AffectState):
        log.debug("%-12s %s  %-12s  %-5s %-18s other=%s self=%s expressed=%s",
                  event.schema, when, "", "", "", axes(event.other), axes(event.self_),
                  axes(event.expressed))
    else:
        log.debug("%-12s %s  %r", event.schema, when, event)

    # Full dataclass
    # log.debug("%-12s %s", event.schema, event)
    
# Get the root path and data paths
#

def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"

# Helper to create an AffectVector
#

def gen_affect_vector(rep: AffectRepresentation, **magnitudes: float) -> AffectVector:
    values = dict.fromkeys(rep.axes, rep.rest)
    for axis, magnitude in magnitudes.items():
        if axis not in values:
            raise ValueError(f"{axis!r} is not an axis of {rep.id}: {rep.axes}")
        values[axis] = magnitude
    return AffectVector(representation=rep.id, values=values)

# Helper to create an Affect Model
#

def build_model(unstated_confidence: float = 0.5, half_life_s: float = 45.0) -> AffectModel:
    """The affect model, wired as `cli.py` wires it — in one place, so two run cells cannot drift.

    The two parameters are the ones worth varying from a notebook. `unstated_confidence` matters
    more than it looks: the keyword decoder leaves `confidence` as None on every observation, so
    this value IS the fold weight for a whole rule-decoder run — at 0.5 the belief moves half way
    to each reading and never further. `refresh_above` stays fixed because nothing can vary it
    until a decoder states a confidence.

    Defaults here, unlike in the package, which refuses them: a defaulted parameter is one that can
    go missing from a run manifest, and a notebook writes no manifest.
    """
    return AffectModel(representation=EKMAN6,
                       policies={Target.OTHER: ConfidenceWeighted(unstated_confidence=unstated_confidence,
                                                                  max_weight=1.0,
                                                                  refresh_above=0.5),
                                 Target.SELF: Assign(),
                                 Target.EXPRESSED: Assign(),
                                 },
                       half_lives={Target.OTHER: half_life_s, Target.SELF: half_life_s},
                       started_at=utc_now(),
                       )


class Collector:
    """Keeps every published record, split by type — the notebook's stand-in for the recorder.

    Registered *alongside* `log_event` because the two do different jobs: the log is for watching
    a run, this is for measuring it. It keeps whole records rather than extracted fields, since
    scoring wants full precision and every axis — both of which `log_event` deliberately discards.

    Step 12 replaces this with `recording/streams.py`, writing the same records as JSONL plus a
    manifest. What the manifest adds is *attribution* rather than collection: `design_version`, the
    resolved config, content hashes and a per-stream record count. Until then a score from here is
    exploratory and is not reproducible from the repository alone.
    """

    def __init__(self) -> None:
        self.utterances: list[Utterance] = []
        self.evidence: list[AffectEvidence] = []
        self.states: list[AffectState] = []

    def __call__(self, event: Event) -> None:
        if isinstance(event, Utterance):
            self.utterances.append(event)
        elif isinstance(event, AffectEvidence):
            self.evidence.append(event)
        elif isinstance(event, AffectState):
            self.states.append(event)

# Text Console

In [ ]:
# Establish the source, decoder and a simple EKMAN6 keyword decoder
# source = TextConsole()
source = TextConsole(read=read_or_end)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)
model = build_model()

# No recorder yet, so a debug observer stands in for one
observers = Observers()
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)

## Test Text

In [2]:
# Load the standard benchmark frames
#
# Written by benchmark_datasets.ipynb: source, split, id, text, then one float column per
# axis of the representation. Nothing is reshaped here — that is what the standard shape is
# for. Each file also carries its own provenance in attrs, including any axis its corpus
# never annotated, which the replay source below insists you acknowledge.

import pandas as pd

simple_bench = pd.read_parquet(DATA_IN / "bench_simple_ekman6.parquet")
brighter_bench = pd.read_parquet(DATA_IN / "bench_brighter_eng.parquet")

print(simple_bench.attrs)
print(brighter_bench.attrs)
simple_bench.head()

{'corpus': 'simple-ekman6', 'representation': 'ekman6/1', 'axis_map': {}, 'unannotated_axes': [], 'rows': 20}
{'corpus': 'brighter-eng', 'representation': 'ekman6/1', 'axis_map': {'joy': 'happiness'}, 'unannotated_axes': ['disgust'], 'rows': 8522}


,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,simple-ekman6,all,simple-ekman6_00000,I am so happy,0.0,0.0,0.0,1.0,0.0,0.0
1,simple-ekman6,all,simple-ekman6_00001,I'm absolutely delighted,0.0,0.0,0.0,1.0,0.0,0.0
2,simple-ekman6,all,simple-ekman6_00002,I was gutted,0.0,0.0,0.0,0.0,1.0,0.0
3,simple-ekman6,all,simple-ekman6_00003,I feel miserable today,0.0,0.0,0.0,0.0,1.0,0.0
4,simple-ekman6,all,simple-ekman6_00004,that is absolutely revolting,0.0,1.0,0.0,0.0,0.0,0.0


In [3]:
# An input source that replays a benchmark frame as utterances carrying ground truth
# For evaluating the ASA pipeline

import asyncio
from collections.abc import AsyncIterator, Iterable


class UtterancesFromDF:
    """Replays a benchmark frame as Utterances carrying their intended affect.

    Knows nothing about any particular corpus: it reads ``text`` and the axis columns of the
    representation it is given, which is exactly what the standard benchmark shape
    guarantees are there.

    ``intended`` is the ground truth a decoder is scored against. ``Utterance``'s own
    docstring names a labelled benchmark set as one of only two sources entitled to set it,
    so this is the sanctioned route rather than a test fixture leaking into the runtime.

    ``assume_absent`` names axes whose corpus does not annotate them; any *other* unlabelled
    axis raises. The frames keep an unannotated axis as NaN because ``rest`` is not
    "unknown" — it is the positive claim that the emotion is absent — so somebody has to
    make that claim, and it should be whoever runs the replay, out loud. BRIGHTER-eng needs
    ``assume_absent=("disgust",)``; its ``attrs`` says so.

    ``gap`` is what makes this a *source* rather than a list. A real participant speaks with
    pauses, and an async generator that never awaits would submit the whole frame before the
    consumer ran once — the published order would then be an artefact of the fake rather
    than the pipeline's behaviour.
    """

    def __init__(self,
                 source_df: pd.DataFrame,
                 representation: AffectRepresentation,
                 *,
                 source: str = "input:replay",
                 gap: float = 1.0,
                 assume_absent: Iterable[str] = ()) -> None:
        self._source_df = source_df
        self._rep = representation
        self._source = source
        self._gap = gap
        self._absent = {str(axis) for axis in assume_absent}

    def _intended(self, row: pd.Series) -> AffectVector:
        """The row's labels as a vector; an unmeasured axis is refused unless assumed absent."""
        magnitudes = {}
        for axis in self._rep.axes:
            magnitude = row[axis]
            if pd.isna(magnitude):
                if str(axis) not in self._absent:
                    raise ValueError(f"{axis} is unlabelled in row {row['id']!r} — name it in "
                                     f"assume_absent to assert the corpus means it is absent")
                continue                  # gen_affect_vector's seeded rest supplies it
            magnitudes[str(axis)] = float(magnitude)
        return gen_affect_vector(self._rep, **magnitudes)

    async def events(self) -> AsyncIterator[Utterance]:
        for _, row in self._source_df.iterrows():
            yield Utterance(text=str(row["text"]),
                            source=self._source,
                            intended=self._intended(row))
            await asyncio.sleep(self._gap)

## Run, Replay

In [4]:
# Establish the replay source, and a simple EKMAN6 keyword decoder
#
# Sliced at the call site rather than with a limit= parameter: the frame is already the
# right object to subset, and 8522 BRIGHTER rows at gap=0.5 would run for 71 minutes. For
# BRIGHTER, the unannotated axis has to be acknowledged:
#   UtterancesFromDF(brighter_bench.head(20), EKMAN6, gap=0.5, assume_absent=("disgust",))

source = UtterancesFromDF(source_df=simple_bench, representation=EKMAN6, gap=0.5)
decoder = KeywordDecoder(representation=EKMAN6, table=EKMAN6_KEYWORDS)
model = build_model()

# No recorder yet: `log_event` stands in for one to watch the run, `records` to measure it
records = Collector()
observers = Observers()
observers.register(records)
observers.register(log_event)

# Kick-off the agent pipeline
await run_agent(source=source, decoder=decoder, state_writer=model.observe, observers=observers)

DEBUG: asa.observer.debug.log_event.line_58 - utterance/1  16:37:38.479  90e27ac8f383        input:replay       'I am so happy' intended={'happiness': 1.0}
DEBUG: asa.observer.debug.log_event.line_62 - evidence/1   16:37:38.479  90e27ac8f383  other decoder:rule       {'happiness': 0.7}
DEBUG: asa.observer.debug.log_event.line_66 - state/2      16:37:38.479                                         other={'happiness': 0.35} self={} expressed={}
DEBUG: asa.observer.debug.log_event.line_58 - utterance/1  16:37:38.982  7d6208cf8cf3        input:replay       "I'm absolutely delighted" intended={'happiness': 1.0}
DEBUG: asa.observer.debug.log_event.line_62 - evidence/1   16:37:38.982  7d6208cf8cf3  other decoder:rule       {'happiness': 0.9}
DEBUG: asa.observer.debug.log_event.line_66 - state/2      16:37:38.982                                         other={'happiness': 0.624} self={} expressed={}
DEBUG: asa.observer.debug.log_event.line_58 - utterance/1  16:37:39.486  541e5eb581d7        inp

## Score — the decoder against the ground truth the replay carried

In [5]:
# Pair each decode with the ground truth its utterance carried
#
# The join key is `of_input`: an AffectEvidence names the Utterance it came from, and that is the
# only thing linking a decode back to an intended vector. A row whose `intended` is None cannot be
# scored and must not enter the denominator — a person typing declares no intent.

# A dropped observer is SILENT by design: a failing observer is caught and dropped so a broken
# recorder cannot stop the agent. One evidence row per utterance is the cheapest check there is.
assert len(records.evidence) == len(records.utterances), (
    f"{len(records.evidence)} evidence rows from {len(records.utterances)} utterances — "
    "an observer that raised would lose rows without saying so")

# Which axes this corpus never annotated, taken from the run itself rather than restated here.
# Two independent records of the same fact, so they can be checked against each other: the source
# holds what it was TOLD to assume absent, the frame holds what the corpus DECLARES. A mismatch
# means either the run silenced an axis that is actually labelled, or the corpus grew one.
unmeasured = tuple(sorted(source._absent))
declared = tuple(sorted(source._source_df.attrs.get("unannotated_axes", ())))
assert unmeasured == declared, (
    f"run assumes {unmeasured} absent, corpus declares {declared} unannotated")

truth = {u.id: u for u in records.utterances}

paired = pd.DataFrame([
    {"id": ev.of_input,
     "text": truth[ev.of_input].text,
     "rationale": ev.rationale,
     **{f"true_{axis}": truth[ev.of_input].intended.values[axis] for axis in EKMAN6.axes},
     **{f"pred_{axis}": ev.affect.values[axis] for axis in EKMAN6.axes}}
    for ev in records.evidence if truth[ev.of_input].intended is not None
])

print(f"{len(paired)} of {len(records.evidence)} rows carry ground truth "
      f"({len(records.evidence) - len(paired)} carried no intended vector)")
print(f"unmeasured axes, excluded from any score: {unmeasured or 'none'}")
paired.head()

20 of 20 rows carry ground truth (0 carried no intended vector)
unmeasured axes, excluded from any score: none


,id,text,rationale,true_anger,true_disgust,true_fear,true_happiness,true_sadness,true_surprise,pred_anger,pred_disgust,pred_fear,pred_happiness,pred_sadness,pred_surprise
0,90e27ac8f383,I am so happy,matched: happiness=happy,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.7,0.0,0.0
1,7d6208cf8cf3,I'm absolutely delighted,matched: happiness=delighted,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.9,0.0,0.0
2,541e5eb581d7,I was gutted,matched: sadness=gutted,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.8,0.0
3,599c86ab64b1,I feel miserable today,matched: sadness=miserable,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.9,0.0
4,a3b4ad651835,that is absolutely revolting,matched: disgust=revolting,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,0.0,0.0


In [6]:
# Dominant-axis agreement, with every reason a row cannot be scored kept as its own outcome
#
# NOT the score. Open question 16 has not settled which of per-axis error and dominant-axis
# agreement is the headline figure, nor how a tie resolves — "delighted but astonished" produces
# one. Reporting the awkward cases separately is what keeps that question visible instead of
# answering it by accident inside a max().
#
# Three outcomes are not errors and must never be counted as one:
#   tie                  two axes at the same magnitude — question 16's unresolved case
#   neutral              nothing fired: no keyword matched, or nothing was labelled
#   no-labelled-emotion  as neutral, but this corpus leaves an axis unannotated, so the sentence
#                        may have carried an emotion nobody looked for. Only ever on the truth side
#   unscorable           the DECODER named an axis the corpus never labelled. BRIGHTER-eng labels
#                        no disgust in any of its 8522 rows while the keyword table has disgust
#                        words, so counting these as false positives would score predictions
#                        against a class nobody annotated — inflating the error rate on a corpus
#                        that never claimed to test it

def dominant(row, prefix: str) -> str:
    """The strongest axis, or a named reason there is not one."""
    scores = {str(axis): row[f"{prefix}_{axis}"] for axis in EKMAN6.axes}
    top = max(scores.values())

    if top == EKMAN6.rest:
        # On the truth side, "nothing fired" is only neutral if every axis was actually looked at
        return "no-labelled-emotion" if (prefix == "true" and unmeasured) else "neutral"

    winners = [axis for axis, value in scores.items() if value == top]
    if any(axis in unmeasured for axis in winners):
        return "unscorable"
    return winners[0] if len(winners) == 1 else "tie"


paired["true_top"] = paired.apply(dominant, axis=1, prefix="true")
paired["pred_top"] = paired.apply(dominant, axis=1, prefix="pred")

agreement = paired.groupby(["true_top", "pred_top"]).size().unstack(fill_value=0)
print(agreement)

# A tie counts on EITHER side, and on BRIGHTER it is mostly the TRUTH that ties: those rows carry
# several emotions at equal strength, so the ground truth has no single dominant axis to agree with.
aside = (paired["pred_top"].isin(("unscorable", "tie"))
         | paired["true_top"].isin(("tie", "no-labelled-emotion")))

print(f"\n{(~aside).sum()} rows scorable, {aside.sum()} set aside:"
      f"\n  {paired['true_top'].eq('tie').sum():4} ground truth has no single dominant axis"
      f"\n  {paired['pred_top'].eq('tie').sum():4} decoder tied between axes"
      f"\n  {paired['pred_top'].eq('unscorable').sum():4} decoder named an axis nobody labelled"
      f"\n  {paired['true_top'].eq('no-labelled-emotion').sum():4} nothing labelled, and an axis "
      "was never looked at")

pred_top   anger  disgust  fear  happiness  neutral  sadness  surprise  tie
true_top                                                                   
anger          2        0     0          0        0        0         0    0
disgust        0        2     0          0        0        0         0    0
fear           0        0     2          0        1        0         0    0
happiness      0        0     0          2        1        0         0    1
neutral        0        0     0          0        3        0         0    0
sadness        0        0     0          1        1        2         0    0
surprise       0        0     0          0        0        0         2    0

19 rows scorable, 1 set aside:
     0 ground truth has no single dominant axis
     1 decoder tied between axes
     0 decoder named an axis nobody labelled
     0 nothing labelled, and an axis was never looked at
